In [52]:
from pathlib import Path
import pandas as pd
import numpy as np
import joblib
BASE_DIR = Path.cwd()
DATA_FILE = BASE_DIR / "campuspulse_akgec_cleaned.csv"
CLUSTER_FILE = BASE_DIR / "cluster_labels.csv"
CLASSIFIER_FILE = BASE_DIR / "best_priority_classifier.pkl"
PREPROCESSOR_FILE = BASE_DIR / "campuspulse_preprocessor.pkl"
OUTPUT_FILE = BASE_DIR / "campuspulse_ranked_complaints.csv"

print("Project folder:", BASE_DIR)


Project folder: c:\Users\v2126\OneDrive\Desktop\CampusPulse\Ensemble_Learning


In [53]:
CLUSTER_FILE = BASE_DIR.parent / "campuspulse_kmeans_clusters.csv"
clusters = pd.read_csv(CLUSTER_FILE)
print("Cluster file shape:", clusters.shape)
print("Columns:", clusters.columns.tolist())
print(clusters.head())

Cluster file shape: (200000, 2)
Columns: ['complaint_id', 'cluster']
  complaint_id  cluster
0   CP-0000001        1
1   CP-0000002        1
2   CP-0000003        1
3   CP-0000004        0
4   CP-0000005        1


In [54]:
DATA_FILE = BASE_DIR.parent / "campuspulse_akgec_cleaned.csv"
data = pd.read_csv(DATA_FILE)
print("Dataset shape:", data.shape)
print("Columns:", data.columns.tolist())
print(data[['complaint_id', 'priority']].head())

Dataset shape: (200000, 37)
Columns: ['complaint_id', 'complaint_date', 'complaint_time', 'complainant_type', 'department', 'year_of_study', 'campus_zone', 'building', 'floor', 'room_lab_no', 'facility_type', 'class_section', 'issue_category', 'issue_subcategory', 'severity', 'affected_users', 'issue_duration_hours', 'previous_similar_complaints', 'complaints_last_7_days', 'complaints_last_30_days', 'recurrence_count', 'equipment_age_years', 'infrastructure_age_years', 'connectivity_impact', 'safety_risk', 'academic_impact', 'operational_impact', 'estimated_repair_cost', 'weather_condition', 'submission_channel', 'complaint_status', 'response_time_hours', 'resolution_time_hours', 'maintenance_team', 'resolution_type', 'priority', 'actual_expenditure']
  complaint_id priority
0   CP-0000001   Medium
1   CP-0000002      Low
2   CP-0000003   Medium
3   CP-0000004      Low
4   CP-0000005     High


In [55]:

data = data.merge(
    clusters[['complaint_id', 'cluster']],
    on='complaint_id',
    how='left'
)
print("Merged dataset shape:", data.shape)
print("Missing cluster labels:", data['cluster'].isna().sum())
print(data[['complaint_id', 'priority', 'cluster']].head())

Merged dataset shape: (200000, 38)
Missing cluster labels: 0
  complaint_id priority  cluster
0   CP-0000001   Medium        1
1   CP-0000002      Low        1
2   CP-0000003   Medium        1
3   CP-0000004      Low        0
4   CP-0000005     High        1


In [56]:
MODEL_FILE = BASE_DIR.parent / "ml" / "models" / "classification" / "final_priority_model.pkl"
PREPROCESSOR_FILE = BASE_DIR.parent / "ml" / "models" / "classification" / "campuspulse_preprocessor.pkl"
model = joblib.load(MODEL_FILE)
preprocessor = joblib.load(PREPROCESSOR_FILE)

print("Model loaded:", type(model).__name__)
print("Preprocessor loaded:", type(preprocessor).__name__)


Model loaded: LogisticRegression
Preprocessor loaded: ColumnTransformer


In [57]:
print("Model expects features:", model.n_features_in_)

if hasattr(preprocessor, "feature_names_in_"):
    print("Preprocessor input features:")
    print(preprocessor.feature_names_in_.tolist())
else:
    print("Preprocessor does not expose feature_names_in_")

Model expects features: 152
Preprocessor input features:
['complainant_type', 'department', 'year_of_study', 'campus_zone', 'building', 'floor', 'room_lab_no', 'facility_type', 'class_section', 'issue_category', 'issue_subcategory', 'severity', 'affected_users', 'issue_duration_hours', 'previous_similar_complaints', 'complaints_last_7_days', 'complaints_last_30_days', 'recurrence_count', 'equipment_age_years', 'infrastructure_age_years', 'connectivity_impact', 'safety_risk', 'academic_impact', 'operational_impact', 'estimated_repair_cost', 'weather_condition', 'submission_channel', 'complaint_month', 'complaint_dayofweek', 'complaint_year', 'is_weekend', 'complaint_hour', 'time_period', 'total_recent_complaints', 'complaint_growth', 'impact_score', 'duration_per_user', 'repair_cost_per_user', 'equipment_infrastructure_age_gap', 'operational_pressure']


In [58]:
def prepare_features(df):
    df = df.copy()
    df["complaint_date"] = pd.to_datetime(
        df["complaint_date"], errors="coerce"
    )
    df["complaint_time"] = pd.to_datetime(
        df["complaint_time"].astype(str), errors="coerce"
    )

    df["complaint_month"] = df["complaint_date"].dt.month
    df["complaint_dayofweek"] = df["complaint_date"].dt.dayofweek
    df["complaint_year"] = df["complaint_date"].dt.year
    df["is_weekend"] = (df["complaint_dayofweek"] >= 5).astype(int)
    df["complaint_hour"] = df["complaint_time"].dt.hour

    df["time_period"] = pd.cut(
        df["complaint_hour"],
        bins=[-1, 5, 11, 17, 21, 24],
        labels=["Night", "Morning", "Afternoon", "Evening", "Late_Night"]
    )

    df["total_recent_complaints"] = (
        df["complaints_last_7_days"].fillna(0)
        + df["complaints_last_30_days"].fillna(0)
    )
    df["complaint_growth"] = (
        df["complaints_last_7_days"].fillna(0)
        / (df["complaints_last_30_days"].fillna(0) + 1)
    )
    df["impact_score"] = (
        df["affected_users"].fillna(0)
        * (1 + df["recurrence_count"].fillna(0))
    )
    df["duration_per_user"] = (
        df["issue_duration_hours"].fillna(0)
        / (df["affected_users"].fillna(0) + 1)
    )
    df["repair_cost_per_user"] = (
        df["estimated_repair_cost"].fillna(0)
        / (df["affected_users"].fillna(0) + 1)
    )
    df["equipment_infrastructure_age_gap"] = (
        df["infrastructure_age_years"].fillna(0)
        - df["equipment_age_years"].fillna(0)
    )
    df["operational_pressure"] = (
        df["complaints_last_7_days"].fillna(0)
        + df["recurrence_count"].fillna(0)
        + df["affected_users"].fillna(0) / 10
    )

    return df




In [59]:


PCA_FILE = (
    BASE_DIR.parent / "ml" / "models" / "classification"
    / "campuspulse_pca.pkl"
)

pca = joblib.load(PCA_FILE)

print("Model type:", type(model).__name__)
print("Model expects features:", model.n_features_in_)

print("\nPreprocessor type:", type(preprocessor).__name__)
print("Preprocessor input columns:", len(preprocessor.feature_names_in_))

# Prepare a small sample using the function already defined
sample = prepare_features(data.head(10))
X_sample = sample.loc[:, preprocessor.feature_names_in_]
X_transformed = preprocessor.transform(X_sample)

print("\nPreprocessor output shape:", X_transformed.shape)
print("PCA object type:", type(pca).__name__)

if hasattr(pca, "n_components_"):
    print("PCA fitted components:", pca.n_components_)
if hasattr(pca, "n_features_in_"):
    print("PCA expects input features:", pca.n_features_in_)
if hasattr(pca, "components_"):
    print("PCA output features:", pca.components_.shape[0])


Model type: LogisticRegression
Model expects features: 152

Preprocessor type: ColumnTransformer
Preprocessor input columns: 40

Preprocessor output shape: (10, 304)
PCA object type: PCA
PCA fitted components: 152
PCA expects input features: 304
PCA output features: 152


C:\Users\v2126\AppData\Local\Temp\ipykernel_33668\4251258138.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["complaint_time"] = pd.to_datetime(


In [62]:


def predict_priority(df):
    prepared = prepare_features(df)

    
    X = prepared.loc[:, preprocessor.feature_names_in_]

    X_transformed = preprocessor.transform(X)

    X_reduced = pca.transform(X_transformed)

    prepared["predicted_priority"] = model.predict(X_reduced)

    # Store model confidence when probability estimates are available
    if hasattr(model, "predict_proba"):
        prepared["priority_confidence"] = (
            model.predict_proba(X_reduced).max(axis=1)
        )

    return prepared


print("Prediction function updated successfully.")


Prediction function updated successfully.


In [ ]:


test_predictions = predict_priority(data.head(10))

print(
    test_predictions[
        ["complaint_id", "priority", "predicted_priority", "cluster"]
    ]
)


  complaint_id priority predicted_priority  cluster
0   CP-0000001   Medium             Medium        1
1   CP-0000002      Low             Medium        1
2   CP-0000003   Medium             Medium        1
3   CP-0000004      Low             Medium        0
4   CP-0000005     High               High        1
5   CP-0000006      Low                Low        1
6   CP-0000007   Medium             Medium        1
7   CP-0000008      Low                Low        1
8   CP-0000009     High               High        0
9   CP-0000010   Medium             Medium        1


C:\Users\v2126\AppData\Local\Temp\ipykernel_33668\4251258138.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["complaint_time"] = pd.to_datetime(


In [64]:

# Generate predictions for the full dataset
data = predict_priority(data)

print("Total complaints:", len(data))
print("\nPredicted priority counts:")
print(data["predicted_priority"].value_counts())

print("\nPrediction confidence:")
print(data["priority_confidence"].describe())


C:\Users\v2126\AppData\Local\Temp\ipykernel_33668\4251258138.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["complaint_time"] = pd.to_datetime(


Total complaints: 200000

Predicted priority counts:
predicted_priority
Medium      93845
Low         58413
High        37716
Critical    10026
Name: count, dtype: int64

Prediction confidence:
count    200000.000000
mean          0.637943
std           0.125173
min           0.419712
25%           0.551535
50%           0.574140
75%           0.784412
max           0.986657
Name: priority_confidence, dtype: float64


In [65]:


def normalize_score(series):
    """Convert values to a 0-100 percentile-based score."""
    values = pd.to_numeric(series, errors="coerce")
    return values.rank(pct=True).fillna(0) * 100


def calculate_campuspulse_scores(df):
    result = df.copy()

    required = [
        "predicted_priority", "cluster", "safety_risk",
        "academic_impact", "operational_impact", "affected_users",
        "recurrence_count", "complaints_last_7_days",
        "previous_similar_complaints", "issue_duration_hours"
    ]

    missing = [col for col in required if col not in result.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    # Convert numeric indicators safely
    numeric_cols = [
        "safety_risk", "academic_impact", "operational_impact",
        "affected_users", "recurrence_count",
        "complaints_last_7_days", "previous_similar_complaints",
        "issue_duration_hours"
    ]

    for col in numeric_cols:
        result[col] = pd.to_numeric(result[col], errors="coerce").fillna(0)

    priority_map = {
        "low": 1,
        "medium": 2,
        "high": 3,
        "critical": 4
    }

    priority_weight = (
        result["predicted_priority"]
        .astype(str).str.strip().str.lower()
        .map(priority_map).fillna(1)
    )

    priority_level_score = ((priority_weight - 1) / 3) * 100

    result["Priority_Score"] = (
        0.40 * priority_level_score
        + 0.20 * normalize_score(result["safety_risk"])
        + 0.15 * normalize_score(result["academic_impact"])
        + 0.15 * normalize_score(result["operational_impact"])
        + 0.10 * normalize_score(result["affected_users"])
    ).round(2)

    result["Individual_Maintenance_Score"] = (
        0.30 * normalize_score(result["recurrence_count"])
        + 0.25 * normalize_score(result["complaints_last_7_days"])
        + 0.20 * normalize_score(result["previous_similar_complaints"])
        + 0.25 * normalize_score(result["issue_duration_hours"])
    )

    result["Cluster_Maintenance_Pattern"] = (
        result.groupby("cluster")["Individual_Maintenance_Score"]
        .transform("mean")
    )

    result["Maintenance_Score"] = (
        0.70 * result["Individual_Maintenance_Score"]
        + 0.30 * result["Cluster_Maintenance_Pattern"]
    ).round(2)

    return result


data = calculate_campuspulse_scores(data)

print("Integrated scoring completed.")
print(data[
    ["complaint_id", "predicted_priority", "cluster",
     "Priority_Score", "Maintenance_Score"]
].head(10))


Integrated scoring completed.
  complaint_id predicted_priority  cluster  Priority_Score  Maintenance_Score
0   CP-0000001             Medium        1           39.13              28.03
1   CP-0000002             Medium        1           40.38              46.56
2   CP-0000003             Medium        1           44.28              53.10
3   CP-0000004             Medium        0           38.61              60.72
4   CP-0000005               High        1           60.27              31.87
5   CP-0000006                Low        1           31.48              43.13
6   CP-0000007             Medium        1           41.13              46.55
7   CP-0000008                Low        1           32.46              34.64
8   CP-0000009               High        0           60.14              64.84
9   CP-0000010             Medium        1           41.41              31.75


In [66]:
data = data.sort_values(
    ["Priority_Score", "Maintenance_Score"],
    ascending=[False, False]
).reset_index(drop=True)

data["Complaint_Rank"] = np.arange(1, len(data) + 1)

OUTPUT_FILE = BASE_DIR / "campuspulse_ranked_complaints.csv"
data.to_csv(OUTPUT_FILE, index=False)

print("Ranking completed!")
print("Total complaints:", len(data))
print("Output saved to:", OUTPUT_FILE)

print("\nTop 20 ranked complaints:")
print(data[
    ["Complaint_Rank", "complaint_id", "predicted_priority",
     "cluster", "Priority_Score", "Maintenance_Score"]
].head(20).to_string(index=False))


Ranking completed!
Total complaints: 200000
Output saved to: c:\Users\v2126\OneDrive\Desktop\CampusPulse\Ensemble_Learning\campuspulse_ranked_complaints.csv

Top 20 ranked complaints:
 Complaint_Rank complaint_id predicted_priority  cluster  Priority_Score  Maintenance_Score
              1   CP-0110942           Critical        2            75.0              78.14
              2   CP-0083548           Critical        2            75.0              78.03
              3   CP-0136308           Critical        2            75.0              76.25
              4   CP-0094539           Critical        2            75.0              76.11
              5   CP-0190550           Critical        2            75.0              72.76
              6   CP-0150160           Critical        2            75.0              70.95
              7   CP-0015721           Critical        2            75.0              70.76
              8   CP-0022387           Critical        2            75.0        